The figures of `figures.ipynb` repeated for the Kuroshio Extension. The study `kuroshio_20240305_20260531` covers 135°E to 160°E and 29°N to 44°N with the detection, tracking, axis, and coverage settings of the Gulf Stream study. The box reaches about 6° upstream of the point where the jet leaves the coast at the Boso Peninsula (about 141°E, 35°N), the analog of Cape Hatteras, and 19° downstream. The SSH maps are AVISO's DUACS-MIOST maps with SWOT, which end on 31 December 2025, so the analysis period runs from 5 March 2024 to 31 December 2025 instead of to 1 June 2026. The open-ocean mask leaves out the Sea of Japan instead of the Great Lakes, and the daily axis is the ADT contour whose level the speed picks between 0.60 and 1.20 m (0.10 and 0.60 m for the Gulf Stream).

North of the axis means the left of the flow at the nearest axis point, and south means the right. The two agree where the jet flows east and stay correct inside a meander. Upstream of the Boso Peninsula the Kuroshio runs south of Honshu, where the left of the flow is not north.

What is known:
- North of the Kuroshio Extension lies the cooler, nutrient-rich Kuroshio-Oyashio mixed water region. South of it lies warm, oligotrophic subtropical water.
- Warm-core anticyclones are shed to the north of the jet and cold-core cyclones to the south, as for the Gulf Stream.

Figure 4 of `figures.ipynb`, from Argo salinity, is not repeated: it reads the 1993 to 2026 DUACS eddy study, which exists only for the Gulf Stream.

In [ ]:
from pathlib import Path
from typing import cast
import sys

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.utils.analysis_period import restrict_to_analysis_period
from eddy_tracking.config import load_analysis_config, read_settings, resolve_bronze_dir, resolve_gold_dir, resolve_output_dir

EXPERIMENT = 'kuroshio_20240305_20260531'
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_analysis_config(EXPERIMENT)
eddy_id_settings = read_settings(resolve_output_dir(EXPERIMENT, 'eddy_id'), 'identify-eddies')
axis_settings = read_settings(resolve_output_dir(EXPERIMENT, 'gulf_stream'), 'trace-axis')
plankton_settings = read_settings(resolve_gold_dir(EXPERIMENT, 'eddy_plankton_table.parquet'), 'collocate-plankton')

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    # matplotlib 3.11 warns about its default calligraphic face, Apple Chancery, even though no label uses it
    'mathtext.cal': 'Arial',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

In [ ]:
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.figure import Figure
from matplotlib.lines import Line2D
from eddy_tracking.preprocess.streamline import compute_signed_distance_km
from eddy_tracking.preprocess.tracks import load_track_observations

polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
identity_columns = ['polarity', 'track_id']

streamline = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/streamline.parquet')
eddy_tracks = restrict_to_analysis_period(pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet'), cfg)
target_class = cast(pd.Series, eddy_tracks['polarity']).map(target_classes)
eddy_tracks['is_target'] = eddy_tracks['movement'].eq(target_class)
track_observations = load_track_observations(resolve_output_dir(EXPERIMENT, 'eddy_track'))
eddy_groups = eddy_tracks[identity_columns].assign(group=np.where(eddy_tracks['is_target'], 'target', 'nontarget'))
plankton = restrict_to_analysis_period(pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet'), cfg).merge(eddy_groups, on=identity_columns)

# The crossing day of a target eddy is the first day its center reaches its destination side of that day's axis, the axis the target class is judged against
destination_signs = {'cyclone': -1, 'anticyclone': 1}  # signed distance is positive north of the axis
daily_axes = {date: day.sort_values('point_idx') for date, day in streamline.groupby('date')}
target_observations = track_observations.merge(eddy_tracks.loc[eddy_tracks['is_target'], identity_columns], on=identity_columns).sort_values(identity_columns + ['date'])
target_observations['on_destination'] = target_observations['polarity'].map(destination_signs).to_numpy() * np.array([
    compute_signed_distance_km(daily_axes[date]['lon'].to_numpy(), daily_axes[date]['lat'].to_numpy(), x, y)[0]
    for date, x, y in zip(target_observations['date'], target_observations['center_lon'], target_observations['center_lat'])
]) > 0
crossing_rows = []
for (polarity, track_id), track in target_observations.groupby(identity_columns):
    on_destination = track['on_destination'].to_numpy()
    entries = np.flatnonzero(on_destination[1:] & ~on_destination[:-1])
    if entries.size:
        crossing_rows.append((polarity, track_id, track['date'].iloc[int(entries[0]) + 1]))
crossings = pd.DataFrame(crossing_rows, columns=identity_columns + ['crossing_date'])

# Case study eddies, picked by rule because no one has looked through the Kuroshio eddies by hand: per polarity, the target eddy with the most
# Copernicus CHL composites on the side of its crossing that has fewer, ties going to more composites in all. A composite counts only on a day
# the eddy was detected, because its map is drawn around that day's speed contour
case_composites = plankton.loc[plankton['group'].eq('target') & plankton['CHL'].notna(), identity_columns + ['date']].merge(track_observations[identity_columns + ['date']], on=identity_columns + ['date']).merge(crossings, on=identity_columns)
case_composites['after_crossing'] = case_composites['date'].ge(case_composites['crossing_date'])
side_counts = cast(pd.DataFrame, case_composites.groupby(identity_columns)['after_crossing'].agg(after='sum', total='size'))
side_counts['fewer'] = np.minimum(side_counts['after'], side_counts['total'] - side_counts['after'])
example_ids = {polarity: int(cast(pd.DataFrame, side_counts.loc[polarity]).sort_values(['fewer', 'total'], ascending=False).index[0]) for polarity in polarity_colors}
example_tracks = {
    polarity: cast(pd.DataFrame, track_observations.loc[track_observations['polarity'].eq(polarity) & track_observations['track_id'].eq(example_ids[polarity])]).sort_values('date')
    for polarity in polarity_colors
}
crossing_dates = crossings.set_index(identity_columns)['crossing_date']
crossing_axes = {polarity: (crossing_dates[polarity, track_id], daily_axes[crossing_dates[polarity, track_id]]) for polarity, track_id in example_ids.items()}
print(eddy_tracks.loc[eddy_tracks['is_target']].groupby('polarity').size().rename('target eddies').to_string())
print(f"case study eddies: cyclone {example_ids['cyclone']}, anticyclone {example_ids['anticyclone']}; crossing days {', '.join(f'{k} {v[0]:%Y-%m-%d}' for k, v in crossing_axes.items())}")
display(side_counts.sort_values(['fewer', 'total'], ascending=False).groupby('polarity').head(3))

In [ ]:
from matplotlib.ticker import MaxNLocator
from eddy_tracking.utils.phase_position import add_phase_position, measure_distance_past_axis

N_AGE_BINS = 5
# Bins on each side of the two-part axis, and how far past the axis an eddy must get before its time on the new side starts
N_PHASE_BINS = 2
NEAR_AXIS_KM = 150
N_RADIAL_BINS = 5
MAX_RADIUS = 2
N_BOOTSTRAP = 2000
RANDOM_SEED = 2026
group_styles = {
    'target': {'linestyle': '-', 'linewidth': 1.2, 'marker': 'o', 'markeredgecolor': 'white', 'markeredgewidth': 0.5},
    'nontarget': {'linestyle': (0, (3, 1.5)), 'linewidth': 0.9, 'marker': 's', 'markerfacecolor': 'white', 'markeredgewidth': 0.7, 'alpha': 0.75},
}
bin_edges = np.linspace(0, 1, N_AGE_BINS + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
radial_edges = np.linspace(0, MAX_RADIUS, N_RADIAL_BINS + 1)
phase_edges = np.linspace(0, 2, 2 * N_PHASE_BINS + 1)
phase_centers = (phase_edges[:-1] + phase_edges[1:]) / 2
phase_ticks = ([0, 0.5, 1, 1.5, 2], ['0', '0.5', '1 | 0', '0.5', '1'])
polarity_sides = {'cyclone': ('north', 'south'), 'anticyclone': ('south', 'north')}

# The 0.2 R ring means of CHL and the nine groups for every eddy-composite of the plankton table, each ring needs 3 valid
# pixels and the min_coverage of collocate-plankton. The Gulf Stream study keeps the same file; it takes about two minutes, so it is built once and read after that.
# Delete it after a re-track
rings_path = DATA_DIR / 'gold/eddy_plankton_rings.parquet'
if not rings_path.exists():
    plankton_fields = ['CHL', 'DIATO', 'DINO', 'GREEN', 'HAPTO', 'MICRO', 'NANO', 'PICO', 'PROCHLO', 'PROKAR']
    ring_edges = np.linspace(0, MAX_RADIUS, 2 * N_RADIAL_BINS + 1)
    n_rings = len(ring_edges) - 1
    all_composites = pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet')[identity_columns + ['date']]
    observation_days = track_observations[identity_columns + ['date', 'center_lon', 'center_lat', 'radius_km']].rename(columns={'date': 'day'})
    record_start, record_end = pd.to_datetime(eddy_id_settings['dates'])
    # The 8-day periods of the composites, restarting on 1 January of each year
    periods = []
    for year in range(record_start.year, record_end.year + 1):
        start = pd.Timestamp(year, 1, 1)
        while start.year == year:
            end = min(start + pd.Timedelta(days=7), pd.Timestamp(year, 12, 31))
            if end >= record_start and start <= record_end:
                periods.append((start, end))
            start = end + pd.Timedelta(days=1)
    fields = xr.open_mfdataset(sorted(resolve_bronze_dir(EXPERIMENT, 'plankton').glob('plankton_*.nc')), combine='by_coords')[plankton_fields]
    lon = fields['longitude'].to_numpy()
    lat = fields['latitude'].to_numpy()
    rings = []
    for start, end in periods:
        composites = all_composites.loc[all_composites['date'].between(start, end)]
        if composites.empty:
            continue
        # Each eddy-composite is centered on its detection nearest the composite date, the earlier one on a tie
        candidates = observation_days.loc[observation_days['day'].between(start, end)].merge(composites, on=identity_columns)
        candidates['offset'] = (candidates['day'] - candidates['date']).abs()
        nearest = candidates.sort_values(['offset', 'day']).drop_duplicates(identity_columns)
        composite = fields.sel(time=slice(start, end)).mean('time').load()
        for polarity, track_id, date, center_lon, center_lat, radius_km in zip(nearest['polarity'], nearest['track_id'], nearest['date'], nearest['center_lon'].to_numpy(), nearest['center_lat'].to_numpy(), nearest['radius_km'].to_numpy()):
            half_width = MAX_RADIUS * radius_km / 111.32
            lon_index = np.flatnonzero(np.abs(lon - center_lon) <= half_width / np.cos(np.radians(center_lat)))
            lat_index = np.flatnonzero(np.abs(lat - center_lat) <= half_width)
            lon_grid, lat_grid = np.meshgrid(lon[lon_index], lat[lat_index])
            # Haversine distance from the center
            half_chord = (
                np.sin(np.radians(lat_grid - center_lat) / 2) ** 2
                + np.cos(np.radians(lat_grid)) * np.cos(np.radians(center_lat)) * np.sin(np.radians(lon_grid - center_lon) / 2) ** 2
            )
            ring_index = np.digitize(2 * 6371 * np.arcsin(np.sqrt(half_chord)).ravel() / radius_km, ring_edges) - 1
            inside = ring_index < n_rings
            n_pixels = np.bincount(ring_index[inside], minlength=n_rings)
            ring = {'polarity': polarity, 'track_id': track_id, 'date': date, 'radial_bin': np.arange(n_rings), 'n_pixels': n_pixels}
            box = composite.isel(longitude=lon_index, latitude=lat_index)
            for field in plankton_fields:
                values = box[field].to_numpy().ravel()
                valid = inside & np.isfinite(values)
                n_valid = np.bincount(ring_index[valid], minlength=n_rings)
                covered = (n_valid >= 3) & (n_valid >= plankton_settings['min_coverage'] * n_pixels)
                ring[field] = np.where(covered, np.bincount(ring_index[valid], weights=values[valid], minlength=n_rings) / np.maximum(n_valid, 1), np.nan)
                ring[f'{field}_n_pixels'] = n_valid
            rings.append(pd.DataFrame(ring))
    pd.concat(rings, ignore_index=True).to_parquet(rings_path, index=False)
# Pairs of 0.2 R rings become 0.4 R rings, weighted by their valid pixels
plankton_rings = restrict_to_analysis_period(pd.read_parquet(rings_path), cfg).dropna(subset=['CHL'])
plankton_rings = cast(pd.DataFrame, plankton_rings.assign(radial_bin=plankton_rings['radial_bin'] // 2, CHL=plankton_rings['CHL'] * plankton_rings['CHL_n_pixels']).groupby(identity_columns + ['date', 'radial_bin'])[['CHL', 'CHL_n_pixels']].sum()).reset_index()
plankton_rings['CHL'] /= plankton_rings['CHL_n_pixels']

analysis = plankton[identity_columns + ['group', 'date', 'age_frac']].assign(source='chl', variable='CHL', concentration=plankton['CHL']).sort_values(identity_columns + ['variable', 'date'])
analysis['age_bin'] = np.minimum((analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1)
analysis['change'] = analysis['concentration'] - analysis.groupby(identity_columns + ['variable'])['concentration'].transform('first')
# Target eddies cross the axis, so they also get a position on the two-part axis: the time on the side they started on, then the time on the side they end on from the first day NEAR_AXIS_KM past the axis, each scaled 0 to 1
crossing_tracks = eddy_tracks.loc[eddy_tracks['is_target']]
crossing_observations = measure_distance_past_axis(resolve_output_dir(EXPERIMENT, 'eddy_track'), resolve_output_dir(EXPERIMENT, 'gulf_stream'), crossing_tracks)
target_phases = add_phase_position(cast(pd.DataFrame, analysis.loc[analysis['group'].eq('target')]), crossing_tracks, crossing_observations, NEAR_AXIS_KM)
binned = pd.concat([
    analysis.assign(time_axis='fraction', position_bin=analysis['age_bin']),
    target_phases.assign(time_axis='phase', position_bin=np.minimum((target_phases['phase_position'] * N_PHASE_BINS).astype(int), 2 * N_PHASE_BINS - 1)),
], ignore_index=True)
bin_positions = {'fraction': bin_centers, 'phase': phase_centers}
bin_keys = ['time_axis', 'source', 'variable', 'group'] + identity_columns + ['position_bin']
eddy_bins = cast(pd.Series, binned.groupby(bin_keys)['change'].mean()).reset_index()
n_eddies = binned.groupby(['time_axis', 'source', 'group', 'polarity'])['track_id'].nunique()
rng = np.random.default_rng(RANDOM_SEED)
summary_rows: list[pd.DataFrame] = []
for keys, source_bins in eddy_bins.groupby(['time_axis', 'source', 'group', 'polarity']):
    time_axis, source, group, polarity = cast(tuple[str, str, str, str], keys)
    positions = bin_positions[time_axis]
    eddy_ids = sorted(source_bins['track_id'].unique())
    draws = rng.integers(0, len(eddy_ids), size=(N_BOOTSTRAP, len(eddy_ids)))
    for variable, variable_bins in source_bins.groupby('variable'):
        matrix = variable_bins.pivot(index='track_id', columns='position_bin', values='change').reindex(index=eddy_ids, columns=range(len(positions))).to_numpy(dtype=float)
        counts = np.isfinite(matrix).sum(axis=0)
        means = np.divide(np.nansum(matrix, axis=0), counts, out=np.full(len(positions), np.nan), where=counts > 0)
        sampled = matrix[draws]
        sampled_counts = np.isfinite(sampled).sum(axis=1)
        sampled_means = np.divide(np.nansum(sampled, axis=1), sampled_counts, out=np.full((N_BOOTSTRAP, len(positions)), np.nan), where=sampled_counts > 0)
        low, high = np.nanquantile(sampled_means, [0.025, 0.975], axis=0)
        summary_rows.append(pd.DataFrame({
            'time_axis': time_axis, 'source': source, 'group': group, 'polarity': polarity, 'variable': variable, 'position_bin': range(len(positions)),
            'position': positions, 'mean': means, 'ci_low': low, 'ci_high': high, 'n_eddies': counts,
        }))
lifetime_summary = pd.concat(summary_rows, ignore_index=True)

# The heatmaps put target eddies on the two-part axis and nontarget eddies on fraction of observed track
heatmap_rows = (('target', 'phase'), ('nontarget', 'fraction'))
heatmap_composites = binned.loc[binned['variable'].eq('CHL') & pd.MultiIndex.from_frame(cast(pd.DataFrame, binned[['group', 'time_axis']])).isin(heatmap_rows)]
radial = plankton_rings.merge(cast(pd.DataFrame, heatmap_composites[['time_axis'] + identity_columns + ['group', 'date', 'position_bin']]), on=identity_columns + ['date'])
radial_grid = cast(pd.Series, cast(pd.Series, radial.groupby(['time_axis'] + identity_columns + ['group', 'position_bin', 'radial_bin'])['CHL'].mean()).groupby(['group', 'time_axis', 'polarity', 'position_bin', 'radial_bin']).mean())
chl_norm = Normalize(np.floor(radial_grid.min() / 0.02) * 0.02, np.ceil(radial_grid.max() / 0.02) * 0.02)


def draw_change(ax: Axes, source: str, variable: str, series: tuple[tuple[str, str], ...]) -> None:
    # series are (group, time_axis) pairs; on a panel with the two-part axis, a group on fraction of observed track spans both parts
    on_phase_axis = any(time_axis == 'phase' for _, time_axis in series)
    ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
    if on_phase_axis:
        ax.axvline(1, color='#999999', linewidth=0.6, zorder=1)
    for polarity, color in polarity_colors.items():
        for group, time_axis in series:
            result = lifetime_summary.loc[lifetime_summary['time_axis'].eq(time_axis) & lifetime_summary['group'].eq(group) & lifetime_summary['polarity'].eq(polarity) & lifetime_summary['variable'].eq(variable)].sort_values('position_bin')
            x = result['position'] * (2 if on_phase_axis and time_axis == 'fraction' else 1)
            ax.errorbar(x, result['mean'], yerr=[result['mean'] - result['ci_low'], result['ci_high'] - result['mean']], fmt='none', ecolor=color, alpha=1 if group == 'target' else 0.45, capsize=1.5, elinewidth=0.7, capthick=0.7, zorder=2)
            ax.plot(x, result['mean'], color=color, markersize=3, label=f'{group.capitalize()} {polarity}s (n = {n_eddies[time_axis, source, group, polarity]})', zorder=4 if group == 'target' else 3, **group_styles[group])
    if on_phase_axis:
        ax.xaxis.set_ticks(*phase_ticks)
        ax.set_xlim(0, 2)
    else:
        ax.xaxis.set_ticks(np.linspace(0, 1, 6).tolist())
        ax.set_xlim(0, 1)
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
    ax.yaxis.set_major_locator(locator)
    ticks = cast(np.ndarray, locator.tick_values(*ax.get_ylim()))
    ax.yaxis.set_ticks(ticks.tolist())
    ax.set_ylim(ticks[0], ticks[-1])


def draw_dotted_bracket(ax: Axes, left: float, right: float, base: float, tip: float) -> None:
    # A bracket along the x-axis, with left and right in x-axis units and base and tip in axes heights. Each half is drawn from the middle outward, starting half a dot in, so both ends carry the same dots
    middle = (left + right) / 2
    for end in (left, right):
        ax.add_artist(Line2D([middle, end, end], [base, base, tip], transform=ax.get_xaxis_transform(), color='#222222', linewidth=0.8, linestyle=(0.5, (1, 1.5)), clip_on=False))


# Figure 1
LIFETIME_HEIGHT = 4.15
fig = cast(Figure, plt.figure(figsize=(6.69, LIFETIME_HEIGHT)))


def add_axes_in_inches(left: float, bottom: float, width: float, height: float) -> Axes:
    return fig.add_axes((left / 6.69, bottom / LIFETIME_HEIGHT, width / 6.69, height / LIFETIME_HEIGHT))


# In inches from the lower left corner: (a) on the left with its legend under it, and the (b) heatmaps on the right, target eddies over nontarget eddies
change_ax = add_axes_in_inches(0.62, 1.5, 2.58, 1.85)
draw_change(change_ax, 'chl', 'CHL', (('target', 'phase'), ('nontarget', 'fraction')))
change_ax.set_title('$\\bf{(a)}$ Change in interior CHL since first observation', loc='left', pad=37.5)
change_ax.set_ylabel('$\\Delta$CHL (mg m$^{-3}$)')
change_ax.set_xlabel('Fraction of time on each side of the Kuroshio\n(target eddies)')
# Nontarget eddies never cross, so they keep fraction of observed track, read on the top axis across both parts
top_axis = change_ax.secondary_xaxis('top', functions=(lambda x: np.divide(x, 2), lambda x: np.multiply(x, 2)))
top_axis.xaxis.set_ticks([0, 0.25, 0.5, 0.75, 1], ['0', '0.25', '0.5', '0.75', '1'])
top_axis.tick_params(length=2.5, width=0.6)
top_axis.set_xlabel('Fraction of observed track (nontarget eddies)', labelpad=3)
change_ax.legend(loc='upper left', bbox_to_anchor=(0, -0.342), ncol=2, fontsize=6.8, handlelength=1.5, handletextpad=0.4, borderaxespad=0, labelspacing=0.3, columnspacing=0.5)

heat_cmap = plt.get_cmap('viridis').copy()  # pyright: ignore[reportAttributeAccessIssue]
heat_cmap.set_bad('#e6e6e6')
heat_axes = [[add_axes_in_inches(left, bottom, 1.0, 1.0) for left in (4.45, 5.6)] for bottom in (2.65, 0.95)]
for row, (group, time_axis) in enumerate(heatmap_rows):
    edges = phase_edges if time_axis == 'phase' else bin_edges
    for col, polarity in enumerate(polarity_colors):
        ax = heat_axes[row][col]
        grid = radial_grid.loc[group, time_axis, polarity].unstack('radial_bin').reindex(index=range(len(edges) - 1), columns=range(N_RADIAL_BINS)).to_numpy(dtype=float).T
        ax.pcolormesh(edges, radial_edges, np.ma.masked_invalid(grid), cmap=heat_cmap, norm=chl_norm, edgecolors='white', linewidth=0.5)
        ax.axhline(1, color='#222222', linewidth=0.8, linestyle=(0, (4, 2.5)), zorder=3)
        if time_axis == 'phase':
            ax.xaxis.set_ticks([0, 1, 2], ['0', '1 | 0', '1'])
            # Under each part of the two-part axis, a dotted bracket over the side that part is for this polarity: the first part is the north side for a cyclone and the south side for an anticyclone.
            # The panel is 1 in square, so 1 pt is 1 / 72 of its height and 1 / 36 of an x-axis unit
            for part, side in enumerate(polarity_sides[polarity]):
                draw_dotted_bracket(ax, part + 2 / 36, part + 1 - 2 / 36, -19 / 72, -14.6 / 72)
                ax.text(part + 0.5, -21 / 72, side, transform=ax.get_xaxis_transform(), ha='center', va='top')
        else:
            ax.xaxis.set_ticks([0, 0.5, 1], ['0', '0.5', '1'])
        ax.xaxis.set_ticks(edges.tolist(), minor=True)
        ax.set_xlim(edges[0], edges[-1])
        ax.yaxis.set_ticks([0, 1, 2], ['0', '1', '2'] if col == 0 else [])
        ax.yaxis.set_ticks(radial_edges.tolist(), minor=True)
        ax.tick_params(length=2)
        ax.tick_params(which='minor', length=1.2)
        if row == 0:
            ax.set_title(f'{polarity.capitalize()}s', pad=5, color=polarity_colors[polarity])
        if col == 0:
            ax.text(-0.3, 0.5, group.capitalize(), transform=ax.transAxes, rotation=90, ha='center', va='center')
    heat_axes[row][0].set_xlabel('Fraction of time on each side' if time_axis == 'phase' else 'Fraction of observed track', labelpad=20 if time_axis == 'phase' else 3)
    heat_axes[row][0].xaxis.label.set_x(1.075)
heat_axes[0][0].annotate('$\\bf{(b)}$ CHL around eddies by radius and time', xy=(0, 1), xycoords='axes fraction', xytext=(0, 16), textcoords='offset points', va='bottom')
fig.text(3.93 / 6.69, 2.3 / LIFETIME_HEIGHT, 'Distance from center (speed radii)', rotation=90, ha='center', va='center')
heat_bar = fig.colorbar(ScalarMappable(norm=chl_norm, cmap=heat_cmap), cax=add_axes_in_inches(4.45, 0.38, 2.15, 0.05), orientation='horizontal')
heat_bar.set_label('CHL (mg m$^{-3}$)', labelpad=1)
heat_bar.ax.tick_params(length=2)
heat_bar.ax.xaxis.set_major_locator(MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]))
heat_bar.outline.set_linewidth(0.5)  # pyright: ignore[reportAttributeAccessIssue]
plt.show()
# The counts the caption quotes: target eddies on the two-part axis, and target eddies that never get NEAR_AXIS_KM past the axis
print(cast(pd.Series, n_eddies.loc['phase']).to_string())
print(f"target eddies never {NEAR_AXIS_KM} km past the axis: {crossing_tracks.groupby('polarity').size().sub(crossing_observations.loc[crossing_observations['across_km'].gt(NEAR_AXIS_KM)].drop_duplicates(identity_columns).groupby('polarity').size(), fill_value=0).to_dict()}")

Figure 1. Chlorophyll-a (CHL) in Kuroshio Extension eddies between 5 March 2024 and 31 December 2025, from the Copernicus multi-sensor product averaged over 8-day periods. Target eddies cross the daily Kuroshio Extension axis, cyclones from north to south and anticyclones from south to north; all other tracked eddies are nontarget eddies. (a) Change in interior CHL, the mean inside the speed contour, from each eddy's first observation. Target eddies (solid lines, bottom axis) are on a two-part time axis: 0 to 1 is the time on the side where the eddy was first detected, up to the first day its center is more than 150 km past the axis, and 1 to 2 is the time after that. All 13 target cyclones and 8 target anticyclones get 150 km past the axis. Nontarget eddies (dashed lines, top axis) are binned by the fraction of their observed track. Each eddy is averaged within a bin before the eddies are averaged, and error bars are 95% bootstrap intervals from resampling eddies. (b) Mean CHL in rings 0.4 speed radii wide around the eddy center, on the time axes of (a). The dashed line marks one speed radius, and the brackets name the side of the Kuroshio Extension for each part of the two-part axis.

Goal: Show that the interior CHL, pigments, and/or PFT classes change over an eddy's lifetime depending on where it moves

In [ ]:
from matplotlib.colors import LogNorm

# Figure 2
# Up to five dates on each side of the crossing for each case eddy, spread evenly over its composites on that side. Each map is the eight-day mean
# of Copernicus CHL around the date, the window of the composite
CASE_DATES_PER_SIDE = 5
CASE_BOX_DEG = 4
case_dates = {}
for polarity, track_id in example_ids.items():
    eddy_composites = case_composites.loc[case_composites['polarity'].eq(polarity) & case_composites['track_id'].eq(track_id)].sort_values('date')
    case_dates[polarity] = []
    for after_crossing in (False, True):
        side_dates = eddy_composites.loc[eddy_composites['after_crossing'].eq(after_crossing), 'date']
        picks = np.unique(np.linspace(0, len(side_dates) - 1, min(CASE_DATES_PER_SIDE, len(side_dates))).round().astype(int))
        case_dates[polarity].append(side_dates.iloc[picks].tolist())
plankton_chl = xr.open_mfdataset(sorted(resolve_bronze_dir(EXPERIMENT, 'plankton').glob('plankton_*.nc')), combine='by_coords')['CHL']

# Case-study maps: 1 in maps 1.06 in apart after a 0.66 in column of row labels, a 0.17 in band for the days from the crossing over each map, 0.17 in under it for the position, and 0.22 in for each eddy's title
panel = 1.0
row_height = 0.17 + panel + 0.17
block_height = 0.22 + 2 * row_height
case_height = 0.05 + 2 * block_height + 0.15 + 0.05
case_fig = cast(Figure, plt.figure(figsize=(6.69, case_height)))


# In inches from the top: one block of maps per case eddy under its panel letter, name, and date span in the eddy's color, the side it came from on top
for case_index, (polarity, track_id) in enumerate(example_ids.items()):
    eddy_track = cast(pd.DataFrame, track_observations.loc[track_observations['polarity'].eq(polarity) & track_observations['track_id'].eq(track_id)]).set_index('date')
    snapshots = []
    for row, side_dates in enumerate(case_dates[polarity]):
        for column, day in enumerate(side_dates):
            contour = cast(pd.Series, eddy_track.loc[day])
            contour_lon, contour_lat = np.asarray(contour['contour_lon']), np.asarray(contour['contour_lat'])
            # The track center can sit outside the contour of its own day, so every map centers on the area centroid of the contour
            cross = contour_lon * np.roll(contour_lat, -1) - np.roll(contour_lon, -1) * contour_lat
            center_lon = float(np.sum((contour_lon + np.roll(contour_lon, -1)) * cross) / (3 * cross.sum()))
            center_lat = float(np.sum((contour_lat + np.roll(contour_lat, -1)) * cross) / (3 * cross.sum()))
            # The box reaches 0.05 degrees past the map, just over one cell, so the edge cells fill the frame
            chl = plankton_chl.sel(
                time=slice(day - pd.Timedelta(days=3), day + pd.Timedelta(days=4)),
                longitude=slice(center_lon - CASE_BOX_DEG / 2 - 0.05, center_lon + CASE_BOX_DEG / 2 + 0.05),
                latitude=slice(center_lat - CASE_BOX_DEG / 2 - 0.05, center_lat + CASE_BOX_DEG / 2 + 0.05),
            ).mean('time').compute()
            daily_axis = daily_axes[day]
            snapshots.append((
                row, column, day, center_lon, center_lat, chl,
                np.append(contour_lon, contour_lon[0]), np.append(contour_lat, contour_lat[0]),
                daily_axis['lon'].to_numpy(), daily_axis['lat'].to_numpy(),
            ))
    # One logarithmic scale for all maps of an eddy, from the 2nd to the 98th percentile of the mapped pixels, with ticks at the 1, 2, 5 values of each decade
    low, high = np.nanpercentile(np.concatenate([snapshot[5].to_numpy().ravel() for snapshot in snapshots]), [2, 98])
    ticks = [tick for tick in np.outer(10.0 ** np.arange(-3, 2), [1, 2, 5]).ravel().tolist() if low <= tick <= high]
    if len(ticks) < 2:
        # A scale that holds fewer than two of those values is also labeled at its two ends
        ticks = [float(low)] + ticks + [float(high)]
    norm = LogNorm(low, high)  # pyright: ignore[reportCallIssue]
    block_top = case_height - 0.05 - case_index * (block_height + 0.15)
    # The first and last dates sit once beside the eddy's name, so each map's label keeps only the days from the crossing
    first_day, last_day = snapshots[0][2], snapshots[-1][2]
    span = f'{first_day.day} {first_day:%b}' + ('' if first_day.year == last_day.year else f' {first_day.year}') + f'–{last_day.day} {last_day:%b %Y}'
    case_fig.text(0.06 / 6.69, (block_top - 0.16) / case_height, f'$\\bf{{({"ab"[case_index]})}}$ CHL around the case study {polarity}, {span}', color=polarity_colors[polarity], va='bottom')
    for row, column, day, center_lon, center_lat, chl, contour_x, contour_y, axis_x, axis_y in snapshots:
        bottom = block_top - 0.22 - row * row_height - 0.17 - panel
        ax = case_fig.add_axes(((0.66 + 1.06 * column) / 6.69, bottom / case_height, panel / 6.69, panel / case_height))
        ax.pcolormesh(chl['longitude'], chl['latitude'], np.ma.masked_invalid(chl.to_numpy()), cmap='viridis', norm=norm, shading='nearest', linewidth=0, rasterized=True)
        ax.plot(axis_x, axis_y, color='black', linewidth=0.8, solid_joinstyle='round')
        ax.plot(contour_x, contour_y, color='white', linewidth=1.8, solid_joinstyle='round')
        ax.plot(contour_x, contour_y, color='black', linewidth=0.8, solid_joinstyle='round')
        ax.set(xlim=(center_lon - CASE_BOX_DEG / 2, center_lon + CASE_BOX_DEG / 2), ylim=(center_lat - CASE_BOX_DEG / 2, center_lat + CASE_BOX_DEG / 2), xticks=[], yticks=[])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.5)
        days_from_crossing = (day - crossing_axes[polarity][0]).days
        ax.annotate(f'{days_from_crossing:+d} d'.replace('-', '−'), xy=(0.5, 1), xycoords='axes fraction', xytext=(0, 2), textcoords='offset points', ha='center', va='bottom')
        ax.annotate(f'{center_lon:.1f}°E {center_lat:.1f}°N', xy=(0.5, 0), xycoords='axes fraction', xytext=(0, -2), textcoords='offset points', ha='center', va='top')
        if column == 0:
            ax.text(-0.64 / panel, 0.5, f'{polarity_sides[polarity][row].capitalize()} of\nKE axis', transform=ax.transAxes, ha='left', va='center')
    bar = case_fig.colorbar(ScalarMappable(norm=norm, cmap='viridis'), cax=case_fig.add_axes((6.0 / 6.69, (block_top - 0.22 - row_height - 0.17 - panel) / case_height, 0.08 / 6.69, (row_height + panel) / case_height)))
    bar.set_ticks(ticks, labels=[f'{tick:.2g}' for tick in ticks])
    bar.minorticks_off()
    bar.set_label('CHL (mg m$^{-3}$)')
    bar.ax.tick_params(length=2, width=0.5)
    bar.outline.set_linewidth(0.5)  # pyright: ignore[reportAttributeAccessIssue]
plt.show()

Figure 2. Copernicus CHL around two case study eddies, (a) a target cyclone and (b) a target anticyclone: up to five dates before the eddy center first reached the far side of the Kuroshio Extension (KE) axis (top row) and up to five after (bottom row), spread evenly over the eddy's 8-day composites on that side. No Kuroshio case study eddies have been picked by hand, so each is the target eddy of its polarity whose shorter side of the crossing holds the most composites; the anticyclone has only three composites after its crossing. Each map is a 4° × 4° box centered on the centroid of that day's speed contour and shows the mean CHL over the eight days around the date. The black line with a white edge is the speed contour, and the plain black line is the KE axis of that day. The number over each map is the days since the crossing, and the position under it is the contour centroid. Each eddy has one logarithmic color scale, from the 2nd to the 98th percentile of its mapped pixels; white is cloud. The boxes are square in degrees, which stretches the eddy east to west by about 1.2 to 1.3.

In [ ]:
from matplotlib.colors import to_rgb
from matplotlib.transforms import blended_transform_factory
from scipy.cluster.hierarchy import dendrogram, fcluster, leaves_list
from eddy_tracking.utils.pigment_clusters import DIAGNOSTIC_PIGMENTS, RATIO_PIGMENTS, build_ward_linkage, correlate_ratios, label_clusters, load_target_composites

# Inches: the height of the tree block and the bar block under it
TREE_HEIGHT = 4.45
BAR_HEIGHT = 3.0
BAR_PANEL_HEIGHT = 1.0
PHASE_BIN_WIDTH = 0.2
# Fraction of a bin a bar fills, leaving gaps between bars
BAR_FILL = 0.8
# The Gulf Stream rule, the lowest cut at which every split still separates a new marker group, lands between the joins at 0.4936 and 0.4941 on this tree
CLUSTER_CUT_HEIGHT = 0.494

# The pigment tree of research_paper/pace_sdp.ipynb on the Kuroshio eddies: Ward clustering of the 12 SDP pigment / Tchla ratios, one sample per target eddy-composite interior mean
pigment_labels = {
    'Fuco': 'Fucoxanthin', 'HexFuco': "19'-Hex-fucoxanthin", 'ButFuco': "19'-But-fucoxanthin",
    'Perid': 'Peridinin', 'Chlc12': 'Chlorophyll-c1+c2', 'Chlc3': 'Chlorophyll-c3', 'Allo': 'Alloxanthin',
    'MV_chlb': 'Monovinyl chlorophyll-b', 'Neo': 'Neoxanthin', 'Viola': 'Violaxanthin',
    'Zea': 'Zeaxanthin', 'DV_chla': 'Divinyl chlorophyll-a',
}
accessory = list(RATIO_PIGMENTS)
target_composites = load_target_composites(DATA_DIR, cfg)
correlation = correlate_ratios(target_composites)
_, pigment_linkage = build_ward_linkage(correlation.to_numpy())
cluster_numbers = fcluster(pigment_linkage, t=CLUSTER_CUT_HEIGHT, criterion='distance')
cluster_labels = {cluster.cluster_id: cluster.labels for cluster in label_clusters(pd.Series(cluster_numbers, index=accessory))}
# One color per named cluster in leaf order, single pigments included, which the dendrogram's own coloring leaves gray; a link below the cut takes its cluster's color
named_clusters = dict.fromkeys(cluster_numbers[leaf] for leaf in leaves_list(pigment_linkage) if cluster_labels[cluster_numbers[leaf]])
cluster_colors = dict(zip(named_clusters, ['#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e', '#e6ab02']))
leaf_colors_by_index = [cluster_colors.get(cluster_numbers[leaf], '#444444') for leaf in range(len(accessory))]
first_leaf = list(range(len(accessory)))
for left_child, _, _, _ in pigment_linkage:
    first_leaf.append(first_leaf[int(left_child)])
figure_height = TREE_HEIGHT + BAR_HEIGHT
cluster_fig = cast(Figure, plt.figure(figsize=(6.69, figure_height)))
# The tree has 1.65 in under it for the leaf names and cluster brackets
tree_ax = cluster_fig.add_axes((0.075, (BAR_HEIGHT + 1.65) / figure_height, 0.915, 2.62 / figure_height))
tree = dendrogram(pigment_linkage, ax=tree_ax, labels=[pigment_labels[pigment] for pigment in accessory], leaf_rotation=90, leaf_font_size=8, link_color_func=lambda link: leaf_colors_by_index[first_leaf[link]] if pigment_linkage[link - len(accessory), 2] < CLUSTER_CUT_HEIGHT else '#444444')
tree['leaves_color_list'] = [leaf_colors_by_index[leaf] for leaf in tree['leaves']]
for label, color in zip(tree_ax.xaxis.get_ticklabels(), tree['leaves_color_list']):
    label.set_color(color)
tree_ax.axhline(CLUSTER_CUT_HEIGHT, color='#999999', linewidth=0.6, linestyle=(0, (4, 2.5)))  # pyright: ignore[reportArgumentType]
tree_ax.tick_params(axis='x', length=0)
tree_ax.autoscale(axis='y')
tree_ax.set_ylim(0, tree_ax.get_ylim()[1])
tree_ax.yaxis.set_major_locator(MaxNLocator(nbins=8, steps=[1, 2, 5, 10]))
tree_ax.yaxis.grid(True, color='#e8e8e8', linewidth=0.5)
tree_ax.set_axisbelow(True)
for spine in tree_ax.spines.values():
    spine.set_visible(True)
tree_ax.set_ylabel('Linkage distance')
tree_ax.set_title('$\\bf{(a)}$ Hierarchical clustering of SDP pigment ratios in target eddies', loc='left')
# Under the leaf names of each cluster, a bracket and the groups whose diagnostic pigments sit in it, in figure coordinates below the lowest leaf name
cluster_fig.draw_without_rendering()
renderer = cluster_fig.canvas.get_renderer()  # pyright: ignore[reportAttributeAccessIssue]
leaf_bottom = cluster_fig.transFigure.inverted().transform((0, min(label.get_window_extent(renderer).y0 for label in tree_ax.get_xticklabels())))[1]
point = 1 / 72 / cluster_fig.get_figheight()
bracket_top, bracket_bottom = leaf_bottom - 4 * point, leaf_bottom - 8 * point
below_axis = blended_transform_factory(tree_ax.transData, cluster_fig.transFigure)
for cluster_id in np.unique(cluster_numbers):
    if not cluster_labels[cluster_id]:
        continue
    members = [position for position, leaf in enumerate(tree['leaves']) if cluster_numbers[leaf] == cluster_id]
    # By cluster number, not leaf color. Leaves sit at x = 5, 15, 25, ...
    left, right = 10 * min(members) + 1, 10 * max(members) + 9
    middle = (left + right) / 2
    color = tree['leaves_color_list'][members[0]]
    cluster_fig.add_artist(Line2D([left, left, right, right], [bracket_top, bracket_bottom, bracket_bottom, bracket_top], transform=below_axis, color=color, linewidth=0.8))
    # A one-pigment cluster right of another one drops its name a line, so the two names do not overlap
    neighbor = cluster_numbers[tree['leaves'][min(members) - 1]] if min(members) > 0 else None
    drop = 9 * point if len(members) == 1 and neighbor is not None and cluster_labels[neighbor] and np.count_nonzero(cluster_numbers == neighbor) == 1 else 0
    cluster_fig.add_artist(Line2D([middle, middle], [bracket_bottom, bracket_bottom - 3 * point - drop], transform=below_axis, color=color, linewidth=0.8))
    # 7 pt, the size figures.ipynb needs so neighboring names do not touch at the 6.69 in AGU width
    tree_ax.text(middle, bracket_bottom - 5 * point - drop, ',\n'.join(cluster_labels[cluster_id]), transform=below_axis, ha='center', va='top', color=color, fontsize=7)

# (b, c): the diagnostic pigment concentrations of all target eddies of a polarity, stacked in leaf order, by fraction of time on each side of the Kuroshio.
# Only the marker pigments of the groups named in (a) are stacked; shared pigments (Chlc12, Chlc3, Viola) name no group.
# Pigments of one cluster share the cluster's color, lightened from dark to pale in leaf order
leaf_colors = {accessory[leaf]: color for leaf, color in zip(tree['leaves'], tree['leaves_color_list']) if accessory[leaf] in DIAGNOSTIC_PIGMENTS}
leaf_pigments = list(leaf_colors)
pigment_colors = {}
for color in dict.fromkeys(leaf_colors.values()):
    same_color = [pigment for pigment, leaf_color in leaf_colors.items() if leaf_color == color]
    for pigment, paleness in zip(same_color, np.linspace(0, 0.6, len(same_color)) if len(same_color) > 1 else [0]):
        pigment_colors[pigment] = tuple((1 - paleness) * channel + paleness for channel in to_rgb(color))
# Composites on the two-part time axis of Figure 1: 0 to 1 is the time on the side the eddy started on, 1 to 2 the time on the side it ends on from the first day NEAR_AXIS_KM past the axis
phases = add_phase_position(target_composites, crossing_tracks, crossing_observations, NEAR_AXIS_KM)
phases['phase_bin'] = np.minimum((phases['phase_position'] / PHASE_BIN_WIDTH).astype(int), round(2 / PHASE_BIN_WIDTH) - 1)
# Every interior pixel counts once, so a large eddy weighs more than a small one. An interior mean times its pixel count is the sum over that composite's pixels
pixel_sums = phases[leaf_pigments].mul(phases['n_eddy_pixels'], axis=0).join(phases[['polarity', 'phase_bin', 'n_eddy_pixels']]).groupby(['polarity', 'phase_bin']).sum()
bin_means = pixel_sums[leaf_pigments].div(pixel_sums['n_eddy_pixels'], axis=0)
# Inches from the bottom of the figure for (b), then (c)
bar_axes = [cluster_fig.add_axes((0.5 / 6.69, bottom / figure_height, 4.45 / 6.69, BAR_PANEL_HEIGHT / figure_height)) for bottom in (1.74, 0.42)]
bar_axes[1].sharex(bar_axes[0])
bar_axes[1].sharey(bar_axes[0])
for ax, letter, polarity in zip(bar_axes, 'bc', ('cyclone', 'anticyclone')):
    means = cast(pd.DataFrame, bin_means.loc[polarity])
    centers = (means.index.to_numpy(dtype=float) + 0.5) * PHASE_BIN_WIDTH
    bottom = np.zeros(len(means))
    for pigment in leaf_pigments:
        ax.bar(centers, means[pigment], width=BAR_FILL * PHASE_BIN_WIDTH, bottom=bottom, color=pigment_colors[pigment], linewidth=0, label=pigment_labels[pigment])
        bottom += means[pigment].to_numpy()
    ax.bar(centers, bottom, width=BAR_FILL * PHASE_BIN_WIDTH, fill=False, edgecolor='#222222', linewidth=0.5)
    # Over each part of the two-part axis, a dotted bracket and the side that part is for this polarity. The panel is 1 in tall and 4.45 in wide for 2 x-axis units, so 1 pt is 1 / 72 of its height and 1 / 160.2 of a unit
    for part, side in enumerate(polarity_sides[polarity]):
        draw_dotted_bracket(ax, part + 2.1 / 160.2, part + 1 - 2.1 / 160.2, 60 / 72, 55.6 / 72)
        ax.text(part + 0.5, 62 / 72, f'{side.capitalize()} side', transform=ax.get_xaxis_transform(), ha='center', va='bottom')
    ax.set_title(f'$\\bf{{({letter})}}$ SDP pigment trends over target {polarity} tracks', loc='left')
    ax.set_ylabel('Pigment (mg m$^{-3}$)')
    ax.yaxis.grid(True, color='#e8e8e8', linewidth=0.5)
    ax.set_axisbelow(True)
bar_axes[0].tick_params(labelbottom=False)
bar_axes[1].xaxis.set_ticks([0, 1, 2], ['0', '1 | 0', '1'])
bar_axes[1].set_xlim(0, 2)
bar_axes[1].set_xlabel('Fraction of time on each side of the Kuroshio')
locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
# Room over the tallest bar for the side brackets and their labels
ticks = cast(np.ndarray, locator.tick_values(0, 1.4 * bin_means.sum(axis=1).max()))
bar_axes[1].yaxis.set_ticks(ticks.tolist())
bar_axes[1].set_ylim(ticks[0], ticks[-1])
# Top to bottom in the order the segments stack, so the legend reads like a bar
handles, labels = bar_axes[0].get_legend_handles_labels()
cluster_fig.legend(handles[::-1], labels[::-1], loc='center left', bbox_to_anchor=(5.05 / 6.69, (0.42 + 1.74 + BAR_PANEL_HEIGHT) / 2 / figure_height), handlelength=1, handleheight=1, labelspacing=0.3)
plt.show()
# The counts the caption quotes: eddy-composites in the tree, and target eddies behind the bars
print(f'eddy-composites in (a): {len(target_composites)}; target eddies in (b, c): {phases.groupby("polarity")["track_id"].nunique().to_dict()}')
print('merge heights of the tree:', np.round(pigment_linkage[:, 2], 2).tolist())

Figure 3. (a) Hierarchical clustering of the 12 SDP accessory pigments inside Kuroshio Extension target eddies. Each sample is one of 149 target eddy-composites: the interior-mean pigment divided by the interior-mean total chlorophyll-a (Tchla). Pigments are joined by Ward linkage on the distance √(2(1 − r)), where r is the Pearson correlation of two pigments' ratios across eddy-composites. The dashed line is the cut at 0.494, the lowest level at which every split still separates a new group of diagnostic (marker) pigments (Kramer & Siegel, 2019), the rule that puts the Gulf Stream cut at 0.72. It sits between two joins only 0.0005 apart, peridinin with 19'-hex-fucoxanthin at 0.4941 and the green-algae with the cryptophyte pigments at 0.4936, so a small change in the data can move it. Pigments joined below the cut share a color, and each bracket names the phytoplankton groups whose diagnostic pigments the cluster holds; chlorophyll-c3 is no group's diagnostic pigment, so its cluster has no name. (b, c) Mean interior concentration of the diagnostic pigments in (b) the 13 target cyclones and (c) the 8 target anticyclones, on the two-part time axis of Figure 1, in bins 0.2 wide, stacked in the leaf order of (a) with each cluster in shades of its color. A bin without a bar has no eddy-composite with valid SDP pigments. Pigments shared by several groups (chlorophyll-c1+c2, chlorophyll-c3, violaxanthin) are left out. Each bar is the mean over all interior pixels of the eddy-composites in its bin, so a large eddy weighs more than a small one. The dotted brackets name the side of the Kuroshio Extension for each part of the axis.

## Backup figures

The two backup figures of `figures.ipynb` for the Kuroshio Extension.

In [ ]:
from cartopy.mpl.geoaxes import GeoAxes
from matplotlib.colors import ListedColormap
from matplotlib.figure import SubFigure
from matplotlib.patches import Rectangle
from eddy_tracking.preprocess.swot import index_swot_files_by_date
from eddy_tracking.utils.analysis_period import read_analysis_period

CONTOUR_EVERY_DAYS = 10
lon_range = eddy_id_settings['lon']
lat_range = eddy_id_settings['lat']
age_cmaps = {'cyclone': ListedColormap(plt.get_cmap('Blues')(np.linspace(0.3, 1, 256))), 'anticyclone': ListedColormap(plt.get_cmap('Reds')(np.linspace(0.3, 1, 256)))}  # pyright: ignore[reportAttributeAccessIssue]
swot_by_date = index_swot_files_by_date(resolve_bronze_dir(EXPERIMENT, 'swot_l4_open_ocean'))

# The two case eddies live at different times, so panel (b) shows the day, within either one's analysis-period life, with the most target eddies.
# Panel (a) maps the ADT of the same day
target_keys = eddy_tracks.loc[eddy_tracks['is_target'], identity_columns]
target_days = track_observations.merge(target_keys, on=identity_columns)
analysis_start, analysis_end = read_analysis_period(cfg)
case_days = pd.DatetimeIndex(np.concatenate([track['date'].to_numpy() for track in example_tracks.values()])).unique()
case_days = case_days[(case_days >= analysis_start) & (case_days <= analysis_end)]
snapshot_date = cast(pd.Timestamp, target_days.loc[target_days['date'].isin(case_days)].groupby('date').size().idxmax())
with xr.open_dataset(swot_by_date[snapshot_date.date()]) as ds:
    region_adt = ds['adt_full'].to_numpy()[0]
    sla = ds['sla'].to_numpy()[0]
    lon = ds['longitude'].to_numpy()
    lat = ds['latitude'].to_numpy()

all_lon = np.concatenate([np.concatenate(track['contour_lon'].to_list()) for track in example_tracks.values()])
all_lat = np.concatenate([np.concatenate(track['contour_lat'].to_list()) for track in example_tracks.values()])
# Panel (c) keeps the aspect ratio of panel (b), so the two maps are the same height side by side; the box covers every contour
# with a margin and grows along its short side to reach that ratio, so no contour is clipped
map_aspect = (lat_range[1] - lat_range[0]) / (lon_range[1] - lon_range[0])
CASE_MARGIN_DEG = 1.0
example_lon = [all_lon.min() - CASE_MARGIN_DEG, all_lon.max() + CASE_MARGIN_DEG]
example_lat = [all_lat.min() - CASE_MARGIN_DEG, all_lat.max() + CASE_MARGIN_DEG]
lon_span, lat_span = example_lon[1] - example_lon[0], example_lat[1] - example_lat[0]
if lat_span < lon_span * map_aspect:
    grow = (lon_span * map_aspect - lat_span) / 2
    example_lat = [example_lat[0] - grow, example_lat[1] + grow]
else:
    grow = (lat_span / map_aspect - lon_span) / 2
    example_lon = [example_lon[0] - grow, example_lon[1] + grow]
example_extent = (*example_lon, *example_lat)

fig = plt.figure(figsize=(6.69, 6.8))
region_fig, tracks_fig = cast(list[SubFigure], fig.subfigures(2, 1, height_ratios=(3.75, 3.05)))
ax = cast(GeoAxes, region_fig.add_axes((0.065, 0.06, 0.81, 0.88), projection=ccrs.PlateCarree()))  # pyright: ignore[reportArgumentType]
ax.set_extent([*lon_range, *lat_range], crs=ccrs.PlateCarree())
adt_fill = ax.contourf(lon, lat, region_adt, levels=np.arange(0, 2.0001, 0.05), cmap='RdBu_r', transform=ccrs.PlateCarree(), zorder=1)
ax.add_feature(cfeature.LAND.with_scale('10m'), facecolor='#e8e8e8', zorder=2)
ax.coastlines(resolution='10m', color='#4d4d4d', linewidth=0.5, zorder=3)
for name, name_lon, name_lat in (('Kuroshio Extension', 152, 37.6), ('Subtropical gyre', 150, 31)):
    ax.text(name_lon, name_lat, name, fontsize=9, fontstyle='italic', color='#262626', ha='center', va='center', transform=ccrs.PlateCarree(), zorder=5)
gridlines = ax.gridlines(draw_labels=True, linewidth=0.5, color='#808080', alpha=0.5, xlocs=range(135, 161, 5), ylocs=range(30, 45, 2), zorder=4)
gridlines.top_labels = False
gridlines.right_labels = False
gridlines.xlabel_style = {'size': 8}
gridlines.ylabel_style = {'size': 8}
adt_bar = fig.colorbar(adt_fill, cax=ax.inset_axes((1.02, 0.125, 0.025, 0.75)))
adt_bar.set_ticks([0, 0.5, 1, 1.5, 2], labels=['0', '0.5', '1', '1.5', '2'])
adt_bar.set_label('Absolute dynamic topography (m)')
adt_bar.outline.set_linewidth(0.6)  # pyright: ignore[reportAttributeAccessIssue]
# The inset sits over Japan and the Sea of Japan, which the eddy analysis leaves out
inset = cast(GeoAxes, ax.inset_axes((0.03, 0.68, 0.24, 0.3), projection=ccrs.PlateCarree()))
inset.set_extent([110, 180, 10, 55], crs=ccrs.PlateCarree())
inset.add_feature(cfeature.LAND.with_scale('110m'), facecolor='#e8e8e8', edgecolor='#808080', linewidth=0.5)
inset.add_feature(cfeature.OCEAN.with_scale('110m'), facecolor='#dbe9f6')
inset.add_patch(Rectangle((lon_range[0], lat_range[0]), lon_range[1] - lon_range[0], lat_range[1] - lat_range[0], linewidth=1, edgecolor='red', facecolor='red', alpha=0.25, transform=ccrs.PlateCarree()))
ax.set_title('$\\bf{(a)}$', loc='left')

maps = tracks_fig.add_gridspec(1, 2, left=0.065, right=0.99, bottom=0.25, top=0.93, wspace=0.19)
snapshot_ax = cast(GeoAxes, tracks_fig.add_subplot(maps[0], projection=ccrs.PlateCarree()))
snapshot_ax.set_extent([*lon_range, *lat_range], crs=ccrs.PlateCarree())
sla_mesh = snapshot_ax.pcolormesh(lon, lat, sla, cmap='RdBu_r', vmin=-0.8, vmax=0.8, alpha=0.8, shading='nearest', rasterized=True, transform=ccrs.PlateCarree(), zorder=1)
sla_bar = fig.colorbar(sla_mesh, cax=snapshot_ax.inset_axes((0, -0.2, 1, 0.045)), orientation='horizontal')
sla_bar.set_label('Sea level anomaly (m)')
sla_bar.set_ticks([-0.8, -0.4, 0, 0.4, 0.8], labels=['−0.8', '−0.4', '0', '0.4', '0.8'])
sla_bar.ax.tick_params(length=2)
sla_bar.outline.set_linewidth(0.6)  # pyright: ignore[reportAttributeAccessIssue]
axis = streamline.loc[streamline['date'].eq(snapshot_date)].sort_values('point_idx')
snapshot_ax.plot(axis['lon'], axis['lat'], color='black', linewidth=1.0, transform=ccrs.PlateCarree(), zorder=4)
today = track_observations.loc[track_observations['date'].eq(snapshot_date)].merge(eddy_tracks[identity_columns + ['is_target']], on=identity_columns)
for polarity, contour_lon, contour_lat, is_target in zip(today['polarity'], today['contour_lon'], today['contour_lat'], today['is_target']):
    snapshot_ax.plot(np.append(contour_lon, contour_lon[0]), np.append(contour_lat, contour_lat[0]), color=polarity_colors[polarity], linewidth=1.4 if is_target else 0.7, linestyle='solid' if is_target else (0, (3, 1.5)), transform=ccrs.PlateCarree(), zorder=5)
snapshot_ax.set_title('$\\bf{(b)}$', loc='left')
snapshot_ax.legend(handles=[
    Line2D([], [], color=polarity_colors['cyclone'], linewidth=1.4, label='Cyclone'),
    Line2D([], [], color=polarity_colors['anticyclone'], linewidth=1.4, label='Anticyclone'),
    Line2D([], [], color='#444444', linewidth=1.4, label='Target eddy'),
    Line2D([], [], color='#444444', linewidth=0.7, linestyle=(0, (3, 1.5)), label='Nontarget eddy'),
    Line2D([], [], color='black', linewidth=1.0, label='Kuroshio axis'),
], loc='lower right', frameon=True, framealpha=0.9, edgecolor='none', fontsize=6.5, handlelength=1.8, borderpad=0.3, labelspacing=0.15)

# Panel (c): both example tracks on one map, each contour every CONTOUR_EVERY_DAYS days, colored by age
example_ax = cast(GeoAxes, tracks_fig.add_subplot(maps[1], projection=ccrs.PlateCarree()))
example_ax.set_extent(example_extent, crs=ccrs.PlateCarree())
for polarity, (crossing_date, crossing_axis) in crossing_axes.items():
    example_ax.plot(crossing_axis['lon'], crossing_axis['lat'], color=polarity_colors[polarity], linewidth=1.1, linestyle='--', transform=ccrs.PlateCarree(), zorder=6)
for panel_index, (polarity, track) in enumerate(example_tracks.items()):
    first_date, last_date = track['date'].iloc[0], track['date'].iloc[-1]
    age_days = (track['date'] - first_date).dt.days
    age_norm = Normalize(0, int(age_days.iloc[-1]))
    drawn = track.loc[age_days.mod(CONTOUR_EVERY_DAYS).eq(0) | track['date'].eq(last_date)]
    for contour_lon, contour_lat, age in zip(drawn['contour_lon'], drawn['contour_lat'], age_days.loc[drawn.index]):
        example_ax.plot(np.append(contour_lon, contour_lon[0]), np.append(contour_lat, contour_lat[0]), color=age_cmaps[polarity](age_norm(age)), linewidth=0.7, transform=ccrs.PlateCarree(), zorder=5)
    age_bar = fig.colorbar(ScalarMappable(norm=age_norm, cmap=age_cmaps[polarity]), cax=example_ax.inset_axes((0.53 * panel_index, -0.2, 0.47, 0.045)), orientation='horizontal')
    age_bar.set_label(f'{polarity.capitalize()} age (days)')
    age_bar.ax.xaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 5, 10]))
    age_bar.ax.tick_params(length=2)
    age_bar.outline.set_linewidth(0.6)  # pyright: ignore[reportAttributeAccessIssue]
    here = today.loc[today['polarity'].eq(polarity) & today['track_id'].eq(example_ids[polarity])]
    for contour_lon, contour_lat in zip(here['contour_lon'], here['contour_lat']):
        snapshot_ax.text(contour_lon.max() + 0.2, contour_lat.max(), 'c', fontweight='bold', transform=ccrs.PlateCarree(), zorder=6)
example_ax.set_title('$\\bf{(c)}$', loc='left')
example_ax.legend(handles=[
    Line2D([], [], color=polarity_colors[polarity], linewidth=1.1, linestyle='--', label=f'Axis, {crossing_date.day} {crossing_date:%b %Y}')
    for polarity, (crossing_date, _) in crossing_axes.items()
], loc='lower left', frameon=True, framealpha=0.9, edgecolor='none', fontsize=6.5, handlelength=1.8, borderpad=0.3, labelspacing=0.15)
for ax in (snapshot_ax, example_ax):
    ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#e9e9e9', zorder=2)
    ax.coastlines(resolution='50m', color='#666666', linewidth=0.5, zorder=3)
    gridlines = ax.gridlines(draw_labels=True, linewidth=0.5, color='#bbbbbb', xlocs=range(135, 161, 5), ylocs=range(30, 45, 2), zorder=3)
    gridlines.top_labels = False
    gridlines.right_labels = False
    gridlines.xlabel_style = {'size': 8}
    gridlines.ylabel_style = {'size': 8}
print(f'panels (a, b) date {snapshot_date:%Y-%m-%d}')
plt.show()

Backup figure 1. (a) Absolute dynamic topography across the study region (135°E to 160°E, 29°N to 44°N) on 3 September 2025. (b) Eddy snapshot on the same day, the day with the most target eddies during the lives of the two case study eddies: target eddies have solid outlines, nontarget eddies have dashed outlines, and c marks the case study cyclone; the case study anticyclone had ended by then. (c) Speed contours of the two case study eddies of Figure 2, drawn every 10 days and colored by days since first detection. Each dashed line is the Kuroshio Extension axis on the day that eddy first reached the far side of it, in the eddy's color.

Goal (a): Display the region that we're focusing on

Goal (b, c): Display the eddy tracking and contour identification, which types of eddies the study targets, and an example of eddy track progression over an eddy's lifetime.

In [ ]:
# Season of every eddy-composite behind Figure 1, counted in the bins of Figure 1, to see whether a part of an eddy's life falls mostly in one season
season_names = {12: 'Winter (DJF)', 1: 'Winter (DJF)', 2: 'Winter (DJF)', 3: 'Spring (MAM)', 4: 'Spring (MAM)', 5: 'Spring (MAM)', 6: 'Summer (JJA)', 7: 'Summer (JJA)', 8: 'Summer (JJA)', 9: 'Autumn (SON)', 10: 'Autumn (SON)', 11: 'Autumn (SON)'}
# Colors chosen to stay apart under every color vision deficiency
season_colors = {'Winter (DJF)': '#3366aa', 'Spring (MAM)': '#2a9d6e', 'Summer (JJA)': '#cc9a22', 'Autumn (SON)': '#b04a2b'}
season_counts = cast(pd.DataFrame, heatmap_composites.assign(season=heatmap_composites['date'].dt.month.map(season_names)).groupby(['group', 'polarity', 'position_bin', 'season']).size().unstack('season', fill_value=0)).reindex(columns=list(season_colors), fill_value=0)

season_fig, season_axes = cast(tuple[Figure, np.ndarray], plt.subplots(2, 2, figsize=(6.69, 4.4), layout='constrained'))
for row, (group, time_axis) in enumerate(heatmap_rows):
    edges = phase_edges if time_axis == 'phase' else bin_edges
    centers = (edges[:-1] + edges[1:]) / 2
    for col, polarity in enumerate(polarity_colors):
        ax = cast(Axes, season_axes[row, col])
        counts = cast(pd.DataFrame, season_counts.loc[group, polarity]).reindex(range(len(centers)), fill_value=0)
        bottom = np.zeros(len(centers))
        for season, color in season_colors.items():
            ax.bar(centers, counts[season], width=0.8 * (edges[1] - edges[0]), bottom=bottom, color=color, edgecolor='white', linewidth=0.6, label=season)
            bottom += counts[season].to_numpy()
        for center, total in zip(centers, bottom):
            ax.annotate(f'{total:.0f}', (center, total), xytext=(0, 2), textcoords='offset points', ha='center', va='bottom', color='#333333')
        if time_axis == 'phase':
            ax.axvline(1, color='#999999', linewidth=0.6, zorder=0)
            ax.xaxis.set_ticks(*phase_ticks)
            for position, side in zip((0.25, 0.75), polarity_sides[polarity]):
                ax.annotate(f'{side} side', (position, 0), xycoords='axes fraction', xytext=(0, -14), textcoords='offset points', ha='center', va='top', color=polarity_colors[polarity])
            ax.set_xlabel('Fraction of time on each side of the Kuroshio', labelpad=13)
        else:
            ax.xaxis.set_ticks(np.linspace(0, 1, 6).tolist(), [f'{tick:g}' for tick in np.linspace(0, 1, 6)])
            ax.set_xlabel('Fraction of observed track')
        ax.set_xlim(edges[0], edges[-1])
        ticks = cast(np.ndarray, MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]).tick_values(0, 1.12 * bottom.max()))
        ax.yaxis.set_ticks(ticks.tolist())
        ax.set_ylim(0, ticks[-1])
        ax.tick_params(axis='x', length=0)
        ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
        ax.set_axisbelow(True)
        if col == 0:
            ax.set_ylabel('Eddy-composites')
        n = heatmap_composites.loc[heatmap_composites['group'].eq(group) & heatmap_composites['polarity'].eq(polarity), 'track_id'].nunique()
        ax.set_title(f'$\\bf{{({"abcd"[2 * row + col]})}}$ {group.capitalize()} {polarity}s (n = {n})', loc='left', color=polarity_colors[polarity])
season_fig.legend(*cast(Axes, season_axes[0, 0]).get_legend_handles_labels(), loc='outside upper center', ncol=4, handlelength=1, handleheight=1, columnspacing=1.5)
plt.show()

Backup figure 2. Season of the 8-day eddy-composites behind Figure 1, counted in the time bins of Figure 1: (a, b) target cyclones and anticyclones on the two-part time axis, and (c, d) nontarget cyclones and anticyclones by fraction of observed track. Each bar stacks the number of eddy-composites by season, with the total above it. The figure checks whether a part of an eddy's life falls mostly in one season.